# k-Nearest Neighbours (k-NN)

k-NN is a simple, **instance-based** classifier: to label a new point it looks at the `k` closest training points and takes a **majority vote**. There is no real "training" step - the model just *memorises* the data and does all the work at prediction time (this is why k-NN is called a **lazy learner**).

**Topics covered in this notebook**

1. **Intuition** - majority vote among the `k` nearest neighbours, the distance metric, and why scaling matters.
2. **Train / fit** - store the (scaled) iris data in a `k=5` classifier.
3. **Inspect / interpret** - accuracy, a per-class report, and the effect of choosing `k`.
4. **When to use it** - strengths, weaknesses, and the curse of dimensionality.

## 1. Intuition

*"You are the company you keep."* To classify a new point, find its `k` nearest neighbours in the training set and let them vote - the most common label among those neighbours wins.

**The distance metric.** "Nearest" needs a definition of distance. The default is **Euclidean** (straight-line) distance. For two points $\mathbf{p}$ and $\mathbf{q}$ with $n$ features:

$$d(\mathbf{p}, \mathbf{q}) = \sqrt{\sum_{i=1}^{n} (p_i - q_i)^2}$$

**Why feature scaling is essential.** Because the label is decided purely by distance, a feature measured on a *large* numeric range will dominate that sum and drown out features on a small range. Imagine one feature in centimetres (0-100) and another as a fraction (0-1): the centimetre feature alone would decide almost every neighbour. **Standardising** each feature to zero mean and unit variance,

$$x' = \frac{x - \mu}{\sigma},$$

puts every feature on a comparable footing so each contributes fairly to the distance. Scaling is optional for tree-based models but **mandatory** for distance-based ones like k-NN.

**No real training.** k-NN is **lazy**: `fit()` simply stores the training points. All the computation - measuring distances to every stored point and voting - happens at `predict()` time.

## 2. Train / fit on a Dataset

We classify iris flowers (3 species) from 4 measurements. The recipe: split the data, **scale the features using training statistics only**, then "fit" a `k=5` classifier (which, for k-NN, just memorises the scaled training set).

In [ ]:
from sklearn.datasets import load_iris                 # built-in iris dataset (no download)
from sklearn.model_selection import train_test_split   # hold out part of the data for testing
from sklearn.preprocessing import StandardScaler        # standardise features: (x - mean) / std
from sklearn.neighbors import KNeighborsClassifier      # the k-NN classifier itself
from sklearn.metrics import accuracy_score              # fraction of correct predictions

In [ ]:
# load_iris() returns a Bunch: .data holds the 150x4 feature matrix,
# .target holds the 0/1/2 species label, .feature_names names the 4 columns.
iris = load_iris()

In [ ]:
import pandas as pd  # only used here to eyeball the raw data as a table

# Wrap the raw feature array in a DataFrame purely for readable, labelled display.
df = pd.DataFrame(iris.data, columns=iris.feature_names)

In [ ]:
# First few rows: each row is one flower, each column one measurement in centimetres.
df.head()

In [ ]:
# Summary stats. Note how the columns live on DIFFERENT ranges/spreads (compare the 'std'
# and min/max rows across columns). That mismatch is exactly why we must scale before k-NN:
# without scaling, the widest-ranging feature would dominate the Euclidean distance.
df.describe()

In [ ]:
# X = feature matrix (150, 4); y = integer species labels (150,) with values 0, 1, 2.
X, y = iris.data, iris.target

In [ ]:
# Hold out 20% of the flowers as an unseen test set. random_state fixes the shuffle so the
# split is reproducible every run. We evaluate on X_test/y_test, which the model never saw.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# k-NN relies on distances, so we scale features first (see section 1 for WHY).
# Fit the scaler on TRAIN ONLY -> it learns each feature's mean & std from the training data.
scaler = StandardScaler().fit(X_train)

# Apply those same train statistics to both splits. Using the test set's own stats would
# leak information from the test set into preprocessing, giving an over-optimistic score.
X_train_s = scaler.transform(X_train)   # standardised training features
X_test_s = scaler.transform(X_test)     # standardised test features (train mean/std)

In [ ]:
# Create a k-NN classifier that votes among the 5 nearest neighbours (default: Euclidean
# distance, equal weight per neighbour).
knn = KNeighborsClassifier(n_neighbors=5)

# 'Fitting' a lazy learner just stores the scaled training points and their labels -
# no coefficients are estimated, no iterations run. The real work happens at predict time.
knn.fit(X_train_s, y_train)

## 3. Inspect / interpret

First the headline accuracy on the held-out test set, then a per-class breakdown, then the effect of `k`.

In [ ]:
# For each test flower, k-NN finds its 5 nearest training neighbours and returns the
# majority-vote label. accuracy_score = fraction of test flowers predicted correctly.
preds = knn.predict(X_test_s)
print("accuracy (k=5):", round(accuracy_score(y_test, preds), 3))

In [ ]:
from sklearn.metrics import classification_report

# Per-class precision / recall / f1 - more informative than a single accuracy number,
# especially when classes are imbalanced. 'support' is the count of each class in y_test.
report = classification_report(y_test, preds, target_names=iris.target_names)
print(report)

### Choosing `k` - the bias/variance trade-off

`k` controls how smooth the decision boundary is, and it is the classic **bias/variance trade-off**:

- **Small `k` (e.g. 1)** -> *low bias, high variance*. The boundary hugs individual points and is very sensitive to noise; a single mislabelled or outlier neighbour can flip a prediction. Risk: **overfitting**.
- **Large `k`** -> *high bias, low variance*. The vote averages over many points, giving a smooth, stable boundary - but if `k` is too large it blurs genuine class boundaries and can **underfit**.

A common practical tip is to use an **odd `k`** for two-class problems so the vote can't tie. Sweep a few values and watch how test accuracy responds.

In [ ]:
# Refit k-NN for several values of k and compare held-out accuracy. Iris is small and the
# classes are well separated once scaled, so accuracy stays high across a wide range of k -
# but on messier data this sweep is how you'd locate the sweet spot in the trade-off above.
for k in [1, 3, 5, 7, 15]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)  # store data for this k
    acc = accuracy_score(y_test, m.predict(X_test_s))                # evaluate this k
    print(f"k={k:2d} -> accuracy {acc:.3f}")

## 4. When to Use It

**Good fit when:**

- You have a **small-to-medium dataset** with meaningful distances and relatively **few dimensions**.
- You want a quick, intuitive **baseline** with essentially no training cost.
- The decision boundary is irregular but locally smooth (k-NN is non-linear and non-parametric - it makes no assumption about the boundary's shape).

**Watch out for:**

- **Always scale features first** - distances are meaningless otherwise.
- **Slow, memory-heavy prediction on large data**: every prediction compares the query against (potentially) every stored training point.
- The **curse of dimensionality**: as the number of features grows, points become roughly equidistant, so "nearest" loses meaning and k-NN degrades. Reduce or select features first.
- **Irrelevant features and noise** hurt more than in many models, because they still contribute to the distance.